In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
en_path = "/content/drive/MyDrive/en-pa.txt/NLLB.en-pa.en"
pa_path = "/content/drive/MyDrive/en-pa.txt/NLLB.en-pa.pa"
score_path = "/content/drive/MyDrive/en-pa.txt/NLLB.en-pa.scores"

In [6]:
with open(en_path, encoding = "utf-8") as f_en, \
     open(pa_path, encoding = "utf-8") as f_pa:

     for _ in range(10):
        print("EN:", f_en.readline().strip())
        print("PA:", f_pa.readline().strip())
        print()

EN: And Our Command is but a single (Act),- like the twinkling of an eye [21].
PA: (ਅਤੇ) ਸਾਡਾ (ਕਿਸੇ ਪ੍ਰਕਾਰ ਦਾ ਕੋਈ) ਗੁਣ (ਜਾਂ) ਔਗੁਣ ਨਹੀਂ ਵਿਚਾਰਿਆ (ਭਾਵ ਸਭ ਕੁਝ ਅਖੋਂ ਓਹਲੇ ਕਰ ਦਿਤਾ) ।

EN: He will be peaceful until the day when he dies!"
PA: (ਇਹ ਹਰਿ-ਨਾਮ) ਤੇਰੇ ਨਾਲ ਸਦਾ ਸਾਥ ਬਣਾਈ ਰੱਖੇਗਾ ॥

EN: "Behold, this is the Word that distinguishes (Good from Evil).
PA: (ਦੁੱਖਾਂ ਕਲੇਸ਼ਾਂ ਤੋਂ) ਬਚਣ ਦਾ ਇਹੀ ਪੱਕਾ ਵਸੀਲਾ ਹੈ ॥

EN: Say, "I seek refuge in Lord of the dawn."
PA: ਆਖ ਕਿ ਜਿਸ (ਜੀਵ ਇਸਤਰੀ) ਦੇ ਮੱਥੇ ਉਤੇ ਭਾਗ (ਲਿਖਿਆ ਹੁੰਦਾ ਹੈ)

EN: "But Iblis (did it not); he refused to be with those who made prostration."
PA: ਪਰ ਅਜ਼ਰਾਈਲ ਨ ਸਾਰਾਂ, ਮੂਲੇ ਲੱਧੀਆਂ।

EN: This is the condition of those who belied Our Signs.
PA: ਝੂਠੇ ਸਾਡੀ ਨਗਰੀ ਦੇ ਕਾਂ ਨਿੱਕਲੇ,

EN: Peace! it is until the emergence of dawn.
PA: صلاة الصبح ਨੂੰ ਕਿਵੇਂ ਉਚਾਰਨਾ ਹੈ

EN: This (Qur'an) is a reminder for those who are aware of Allāh.
PA: (ਜਦੋਂ ਕਿ) ਗਿਆਨ ਵਾਲਿਆਂ ਲਈ ਇਹ ਤਤ ਵੀਚਾਰ (ਭਾਵ ਸਾਰ ਸਿਧਾਂਤ ਹੈ) ।

EN: "Do you read the Quran?"
PA: ਕੀ ਤੁਸੀਂ Qutan ਦਾ

EN: nay, but we are d

In [7]:
with open(en_path, encoding="utf-8") as f:
    n = sum(1 for _ in f)

print(f"{n:,} sentence pairs")

11,123,768 sentence pairs


In [8]:
with open(score_path) as f:
    for _ in range(10):
        print(f.readline())

1.2494746

1.2492368

1.2491212

1.2483953

1.2481887

1.248007

1.2479923

1.2479758

1.2476087

1.2475932



In [7]:
MAX_EXAMPLES = 1_000_000

In [8]:
import itertools

with open(en_path, encoding="utf-8") as f_en, \
     open(pa_path, encoding="utf-8") as f_pa:
     
    en_subset = list(itertools.islice(f_en, MAX_EXAMPLES))
    pa_subset = list(itertools.islice(f_pa, MAX_EXAMPLES))

In [9]:
import itertools
import unicodedata
import re

MIN_WORDS = 2
MAX_WORDS = 100

total = 0
empty_removed = 0
length_removed = 0
ratio_removed = 0
script_removed = 0
duplicate_removed = 0
kept = 0

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def gurmukhi_ratio(text):
    total_chars = len(text)

    gurmukhi = sum(
        1 for ch in text
        if '\u0A00' <= ch <= '\u0A7F'
    )

    return gurmukhi / max(total_chars, 1)

seen = set()
ratio_fail_examples = []

with open(en_path, encoding="utf-8") as f_en, \
     open(pa_path, encoding="utf-8") as f_pa, \
     open("clean.en", "w", encoding="utf-8") as out_en, \
     open("clean.pa", "w", encoding="utf-8") as out_pa:

    for en, pa in itertools.islice(zip(f_en, f_pa), MAX_EXAMPLES):

        total += 1

        en = normalize_text(en)
        pa = normalize_text(pa)

        if not en or not pa:
            empty_removed += 1
            continue

        en_len = len(en.split())
        pa_len = len(pa.split())

        if not (MIN_WORDS <= en_len <= MAX_WORDS):
            length_removed += 1
            continue

        if not (MIN_WORDS <= pa_len <= MAX_WORDS):
            length_removed += 1
            continue

        ratio = en_len / pa_len

        if ratio < 0.75 or ratio > 2.5:
            if len(ratio_fail_examples) < 20:
                ratio_fail_examples.append(
                (en, pa, ratio)
            )
            ratio_removed += 1
            continue

        if gurmukhi_ratio(pa) < 0.5:
            script_removed += 1
            continue

        pair_hash = hash((en, pa))

        if pair_hash in seen:
            duplicate_removed += 1
            continue

        seen.add(pair_hash)

        out_en.write(en + "\n")
        out_pa.write(pa + "\n")

        kept += 1

print("\n===== CLEANING REPORT =====")
print(f"Total pairs processed:     {total:,}")
print(f"Kept pairs:                {kept:,}")
print(f"Empty removed:             {empty_removed:,}")
print(f"Length removed:            {length_removed:,}")
print(f"Ratio removed:             {ratio_removed:,}")
print(f"Script removed:            {script_removed:,}")
print(f"Duplicates removed:        {duplicate_removed:,}")

removal_total = (
    empty_removed
    + length_removed
    + ratio_removed
    + script_removed
    + duplicate_removed
)

print(f"Total removed:             {removal_total:,}")
print(f"Retention rate:            {100 * kept / total:.2f}%")


===== CLEANING REPORT =====
Total pairs processed:     1,000,000
Kept pairs:                820,989
Empty removed:             0
Length removed:            4
Ratio removed:             172,952
Script removed:            5,911
Duplicates removed:        144
Total removed:             179,011
Retention rate:            82.10%


In [12]:
for i, ex in enumerate(ratio_fail_examples, 1):
    print(f"Example {i}")
    print(f"Ratio: {ex[2]:.2f}")
    print("EN:", ex[0])
    print("PA:", ex[1])
    print("-" * 80)

Example 1
Ratio: 2.75
EN: nay, but we are deprived (from the results of our work)!"
PA: ਪਰ ਅਸੀਂ ਅੱਡੀਆਂ ਹੋਈਆਂ
--------------------------------------------------------------------------------
Example 2
Ratio: 0.60
EN: This alone is a warning sign.
PA: ਇਹ ਮਨੂਆ ਇਸ ਗੱਲ ਦਾ ਚੇਤਾ (ਧਿਆਨ) ਹੀ ਨਹੀਂ ਕਰਦਾ,
--------------------------------------------------------------------------------
Example 3
Ratio: 0.61
EN: Is the One who can create, equal to those (idols), who cannot create anything?
PA: ਜਿਸ ((ਪ੍ਰਭੂ) ਨੇ (ਜੀਵ) ਪੈਦਾ ਕੀਤੇ ਹਨ (ਮਨਮੁਖ) ਉਸ (ਪ੍ਰਭੂ ਨੂੰ) ਨਹੀਂ ਯਾਦ ਕਰਦੇ (ਅਤੇ) ਬਿਨਾਂ ਸਿਮਰਨ ਤੋਂ ਕਿਵੇਂ ਸੁਖ ਪਾ ਸਕੀਦਾ ਹੈ?
--------------------------------------------------------------------------------
Example 4
Ratio: 0.50
EN: That will surely be a scam.
PA: ਨਹੀਂ ਤਾਂ ਇਹ ਮਨ (ਵਿਕਾਰਾਂ ਵਿਚ ਪੈ ਕੇ) ਬੜਾ ਦੁਖੀ ਹੋਵੇਗਾ ॥
--------------------------------------------------------------------------------
Example 5
Ratio: 0.50
EN: Those are the inheritors.
PA: ਉਹੀ ਸਾਰੇ ਬੰਦੇ (ਅਸਲ) ਭਾਗਾਂ ਵਾਲੇ ਹਨ ॥
----------------------------------

In [3]:
PROJECT_DIR = "/content/drive/MyDrive/en-pa.txt"

In [14]:
import sentencepiece as spm

In [15]:
files = ["clean.en", "clean.pa"]

with open("spm_corpus.txt" , "w", encoding = "utf-8") as outfile:
    for file in files:
        with open(file, "r", encoding="utf-8") as infile:
            for line in infile:
                line = line.strip()
                if line:
                    outfile.write(line+"\n")

In [18]:
spm.SentencePieceTrainer.train(
    input= "spm_corpus.txt",
    model_prefix= f"{PROJECT_DIR}/punjabi_en_bpe",
    vocab_size=32000,
    model_type= "bpe",
    character_coverage=1.0,
    unk_id=0,
    pad_id=1,
    bos_id=2,
    eos_id=3
)

In [20]:
m1 = "/content/drive/MyDrive/en-pa.txt/punjabi_en_bpe.model"
v1 = "/content/drive/MyDrive/en-pa.txt/punjabi_en_bpe.model"

In [40]:
sp = spm.SentencePieceProcessor()
sp.load(m1)

True

In [23]:
text = "ਮੈਂ ਸਕੂਲ ਜਾ ਰਿਹਾ ਹਾਂ"
tokens = sp.encode(text, out_type=str)
print(tokens)

['▁ਮੈਂ', '▁ਸਕੂਲ', '▁ਜਾ', '▁ਰਿਹਾ', '▁ਹਾਂ']


In [24]:
print("Vocabulary Size:",sp.get_piece_size())

Vocabulary Size: 32000


In [25]:
for i in range(10):
    print(i, sp.id_to_piece(i))

0 <unk>
1 <pad>
2 <s>
3 </s>
4 ▁t
5 he
6 ▁a
7 ▁ਹ
8 ▁ਕ
9 ▁ਸ


In [26]:
decoded = sp.decode(tokens)
print(decoded)

ਮੈਂ ਸਕੂਲ ਜਾ ਰਿਹਾ ਹਾਂ


In [27]:
import numpy as np

lengths = []

with open("clean.pa", encoding="utf-8") as f:
    for line in f:
        lengths.append(len(sp.encode(line.strip())))

print("Avegrage length:", np.mean(lengths))
print("Max Length:", np.max(lengths))

Avegrage length: 16.987832967311377
Max Length: 120


In [28]:
lengths_en = []

with open("clean.en", encoding="utf-8") as f:
    for line in f:
        lengths_en.append(len(sp.encode(line.strip())))

print("Avegrage length:", np.mean(lengths_en))
print("Max Length:", np.max(lengths_en))

Avegrage length: 16.89932264622303
Max Length: 132


In [29]:
print(sp.encode("Hello world", out_type=str))


['▁Hello', '▁world']


In [30]:
print(sp.encode("ਵਿੰਗਾ-ਤੜਿੰਗਾ", out_type=str))

['▁ਵਿ', 'ੰਗਾ', '-', 'ਤ', 'ੜ', 'ਿੰ', 'ਗਾ']


In [31]:
spm.SentencePieceTrainer.train(
    input= "spm_corpus.txt",
    model_prefix= f"{PROJECT_DIR}/punjabi_en_bpe1",
    vocab_size=16000,
    model_type= "bpe",
    character_coverage=0.9995,
    unk_id=0,
    pad_id=1,
    bos_id=2,
    eos_id=3
)

In [32]:
m2 = "/content/drive/MyDrive/en-pa.txt/punjabi_en_bpe1.model"

In [33]:
sp.load(m2)

True

In [34]:
for i in range(10):
    print(i, sp.id_to_piece(i))

0 <unk>
1 <pad>
2 <s>
3 </s>
4 ▁t
5 he
6 ▁a
7 ▁ਹ
8 ▁ਕ
9 ▁ਸ


In [35]:
lengths1 = []

with open("clean.pa", encoding="utf-8") as f:
    for line in f:
        lengths1.append(len(sp.encode(line.strip())))

print("Avegrage length:", np.mean(lengths1))
print("Max Length:", np.max(lengths1))

Avegrage length: 17.93526222641229
Max Length: 132


In [36]:
lengths_en1 = []

with open("clean.en", encoding="utf-8") as f:
    for line in f:
        lengths_en1.append(len(sp.encode(line.strip())))

print("Avegrage length:", np.mean(lengths_en1))
print("Max Length:", np.max(lengths_en1))

Avegrage length: 17.921989210574075
Max Length: 134


In [41]:
used_ids = set()

with open("clean.pa", encoding="utf-8") as f:
    for line in f:
        used_ids.update(sp.encode(line.strip()))

print("Used vocab:", len(used_ids))
print("Total vocab:", sp.get_piece_size())
print("Utilization:", len(used_ids)/sp.get_piece_size()*100)

Used vocab: 23680
Total vocab: 32000
Utilization: 74.0


In [42]:
from collections import Counter

counter = Counter()

with open("clean.pa", encoding="utf-8") as f:
    for line in f:
        pieces = sp.encode(line.strip(), out_type=str)

        for p in pieces:
            clean = p.replace("▁", "")
            counter[len(clean)] += 1

total = sum(counter.values())

single_char_pct = 100 * counter[1] / total

print(f"Single-character pieces: {single_char_pct:.2f}%")

Single-character pieces: 13.70%


In [39]:
print(sp.encode("ਵਿੰਗਾ-ਤੜਿੰਗਾ", out_type=str))
print(sp.encode("Hello world", out_type=str))

['▁ਵਿ', 'ੰਗਾ', '-', 'ਤ', 'ੜ', 'ਿੰ', 'ਗਾ']
['▁Hello', '▁world']


In [11]:
import shutil

shutil.copy(
    "clean.en",
    f"{PROJECT_DIR}/clean.en"
)

shutil.copy(
    "clean.pa",
    f"{PROJECT_DIR}/clean.pa"
)

'/content/drive/MyDrive/en-pa.txt/clean.pa'